In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import all_methods_newsvendor as mf
from tqdm import tqdm
from sklearn.metrics import r2_score
import statsmodels.api as sm
import xgboost as xgb
import os

# suppress all warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#load data
feature = np.load("feature.npy")
label = np.load("label.npy")
feature_with_constant = np.ones((feature.shape[0],feature.shape[1],feature.shape[2]+1))
feature_with_constant[:,:,1:] = feature
feature_with_constant.shape

In [ ]:
for N in [15]:
    for b in [0.75,0.8,0.85,0.9,0.95]:
        h = 1 - b
        cost = []
        decision = []
        true_decision = []
        alpha_list = []
        time = []
        cost_name = ['decentralised_ols','shrunken_saa','Random Forest',
                    'shrunken_non_linear','DAC','PAB_linear',
                    'PAB_tree','centralised_ols','shrunken_ols']
        for inital_point in tqdm(range(0,label.shape[1] - N )):
            # print(inital_point)
            test_point = inital_point + N
            
            # obtain valid historical data and test data
            X_hats = []
            y_hats = []
            X_test = []
            y_test = []
            X_PAB = feature_with_constant[:,inital_point:inital_point + N,:]
            y_PAB = label[:,inital_point:inital_point + N]
            for k in range(label.shape[0]):
                X_temp = []
                y_temp = []
                X_test.append(feature_with_constant[k,test_point])
                y_test.append(label[k,test_point])
                for j in range(inital_point,inital_point + N):
                    if label[k,j] != 0:
                        y_temp.append(label[k,j])
                        X_temp.append(feature_with_constant[k,j])
                X_hats.append(np.array(X_temp))
                y_hats.append(np.array(y_temp))
    
            true_decision.append(y_test)
     
            cost_temp = mf.main(X_hats,y_hats, X_test,y_test,X_PAB,y_PAB,h,b, N,inital_point)
   
            cost.append(cost_temp)
        cost = np.array(cost)
        print(np.mean(cost,axis = 0))
       